In [1]:
from molshift.tools.frag_lib_loader import load_fragment_data

whole_smiles = "CC(C)(c1cccc(n1)N2c3c(cnc(n3)Nc4ccc(cc4)N5CCN(CC5)C)C(=O)N2CC=C)O"
scaffold_smiles ="O=C1NNc2nc(Nc3ccccc3)ncc12"
top_n = 100
resources = load_fragment_data()
sim_mode = "desc"
output_dir = f"Adavosertib_scaff_repl/{sim_mode}"


In [2]:
from molshift.tools.scaff_repl import run_scaff_repl
from pathlib import Path

products = run_scaff_repl(
        whole_smiles = whole_smiles,
        scaffold_smiles = scaffold_smiles,
        top_n = top_n,
        resources = resources,
        output_dir = Path(output_dir),
        sim_mode = sim_mode
)

USRCAT cache loaded: 108,953 vectors
5,972 pass attachment-count + heavy-atom window (13.6-25.5 heavy atoms)
493 pass the bond-distance filter
493 of 493 candidates have a USRCAT vector (reference conformers used: 5)
✓ 493 ranked -> Adavosertib_scaff_repl\desc/ranked_scaffolds.csv / .json
100 ranked scaffolds used (top_n=100) → 100 gave products, 0 didn't match, 0 failed to build
✓ 100 products saved → Adavosertib_scaff_repl\desc\scaff_replaced_products.csv


In [3]:
import pandas as pd
scaff_repl_df = pd.read_csv(products['Output_path'])

In [4]:
import json, sys, platform, datetime, pathlib

def save_manifest(params: dict, path=f"{output_dir}/manifest.json"):
    manifest = {
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "params": params,
    }
    pathlib.Path(path).write_text(json.dumps(manifest, indent=2, default=str))
    return manifest

# Example
params = {
    "Whole smiles": whole_smiles,
    "Scaffold smiles": scaffold_smiles,
    "Similarity screening mode": sim_mode,
    "Scaff repl top_n": top_n,
    "Number of scaff_repl": len(scaff_repl_df),
    "Output path": products['Output_path']
}
save_manifest(params)

{'timestamp': '2026-11-15T15:38:53',
 'python': '3.14.2',
 'platform': 'Windows-11-10.0.26200-SP0',
 'params': {'Whole smiles': 'CC(C)(c1cccc(n1)N2c3c(cnc(n3)Nc4ccc(cc4)N5CCN(CC5)C)C(=O)N2CC=C)O',
  'Scaffold smiles': 'O=C1NNc2nc(Nc3ccccc3)ncc12',
  'Similarity screening mode': 'desc',
  'Scaff repl top_n': 100,
  'Number of scaff_repl': 100,
  'Output path': 'Adavosertib_scaff_repl\\desc/scaff_replaced_products.csv'}}

**Filteration**

In [6]:
from molshift.tools.ref_filter import filter_by_reference_parallel
import os

filtered, windows = filter_by_reference_parallel(scaff_repl_df, "product_smiles", whole_smiles,n_jobs = os.cpu_count()-4,
                                      sa_buffer=1.0, qed_buffer=0.1)
filtered.to_csv(Path(output_dir)/"scaff_replaced_filtered.csv", index=False)

Filtering (12 procs): 100%|██████████| 1/1 [00:01<00:00,  1.51s/it]


**Docking**

*Perform redocking first to prepare the protein and the pocket files*

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.disable(logging.WARNING)
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")
from vina_docking.tools.ligand_docking import run_ligand_docking


dock_config_path = r"dock_config.json"
PROT_DIR = r"Docking_Validations\WEE1"
BASE_NAME = "5V5Y"
OUT_DIR = Path(output_dir)/"Docking"
LIGAND_FILE = Path(output_dir)/"scaff_replaced_filtered.csv"
SMILES_COL = "product_smiles"
K = 30
HAS_REFERENCE = True
SCORE_THRESHOLD = 0.5
REF_PATH = r"Docking_Validations\WEE1\REFERENCE"
run_ligand_docking(
        PROT_DIR,
        BASE_NAME,
        OUT_DIR,
        LIGAND_FILE,
        SMILES_COL,
        K,
        HAS_REFERENCE,
        SCORE_THRESHOLD,
        REF_PATH,
        dock_config_path,
    )